# 탐색적 데이터 분석(EDA) — 질문에서 새 질문까지 정답지

이 노트북은 `EDA_Practice.ipynb`의 모든 연습문제(1~12번)에 대한 **모범 풀이 코드와 결과 해석**을 수록한 정답지입니다.
데이터 품질 진단(결측치, 완전 중복, 극단값 탐지), 평균 vs 중앙값 왜곡, 상관관계 vs 인과관계 구분, 시나리오별 민감도 분석 및 한 페이지 종합 보고서를 완벽히 제공합니다.


## 처음 배우는 분을 위한 안내

EDA는 계산이나 그림으로 자료를 살펴 질문을 구체화하는 과정입니다. 데이터 사전은 열 이름과 뜻을 정리한 표입니다. order_id=주문 번호, customer_id=고객 번호, revenue=주문 금액, quantity=수량, unit_price=단가, region=지역, delivery_days=배송일수, customer_rating=평점입니다. customer의 churn은 Churn(이탈)·Stay(유지), usage_count는 이용 횟수입니다.

**학습 순서:** 문제 1~5는 필수 기초, 6~12는 선택 도전입니다. orders_raw는 보존본, orders는 실습 복사본입니다. 동일 주문 ID가 있어 처음부터 한 행을 고유 주문이라고 확정하지 않습니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

## 실행 준비

Python 3와 pandas가 필요합니다. 정렬 노트북의 rrule에는 python-dateutil, EDA의 선택 그래프에는 matplotlib이 필요합니다. 없는 경우 별도 코드 셀에서 `%pip install pandas python-dateutil matplotlib`을 한 번 실행하세요. 이 노트북은 외부 CSV 없이 실습 데이터가 생성됩니다.

Colab에서는 `.ipynb`를 열고 준비 셀부터 실행하세요. 아래 코드는 실습 데이터 생성만 수행합니다.

In [ ]:
import pandas as pd

# 의도적으로 결측값·극단값·동일 ID가 있는 학습용 가상 데이터입니다.
# 실제 고객 데이터의 분석 결과로 일반화하지 마세요.
orders_raw = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 110],
    'customer_id': ['A', 'B', 'A', 'C', 'D', 'B', 'E', 'F', 'G', 'H', 'H'],
    'order_date': ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
                   '2024-01-05', '2024-02-01', '2024-02-02', '2024-02-03',
                   '2024-02-04', '2024-02-05', '2024-02-05'],
    'product_category': ['Food', 'Book', 'Food', 'Tech', 'Book', 'Tech',
                         'Food', 'Book', 'Tech', 'Food', 'Food'],
    'region': ['South', 'North', 'South', 'East', None, 'South',
               'East', 'North', 'South', 'East', 'East'],
    'quantity': [2, 1, 3, 1, 2, 2, 1, 2, 1, 2, 2],
    'unit_price': [100, 200, 100, 1000, 200, 1000, 100, 200, 1000, 100, 100],
    'revenue': [200, 200, 300, 1000, None, 2000, 100, 400, 50000, 200, 200],
    'delivery_days': [1, 3, 2, 8, 4, 7, 9, 3, 12, 2, 2],
    'customer_rating': [4.5, 4.0, 4.8, 2.0, 4.1, 3.0, 2.5, 4.3, 1.0, 4.6, 4.6]
})
orders_raw['order_date'] = pd.to_datetime(orders_raw['order_date'])
orders = orders_raw.copy(deep=True)
customers = pd.DataFrame({
    'customer': list('ABCDEF'), 'usage_count': [3, 4, 2, 18, 15, 20],
    'churn': ['Churn', 'Churn', 'Churn', 'Stay', 'Stay', 'Stay']
})

## 문제 1. 분석 질문 정하기 [필수 기초]

orders의 데이터 사전과 행 단위를 확인한 뒤 매출·배송·고객에 관한 분석 질문을 각각 하나씩 적으세요. 질문마다 필요한 열과 비교 지표를 쓰세요.

**점검 기준:** 질문은 데이터로 확인 가능한 형태여야 하며 한 행을 고유 주문 한 건으로 볼 수 있는지도 먼저 점검합니다.

In [ ]:
# 문제 1 풀이
print("=== orders 데이터 구조 확인 ===")
display(orders.head())
print("\n행 단위: 개별 주문 트랜잭션 기록")
print(f"전체 행 수: {len(orders)}건, 고유 주문 ID 수: {orders['order_id'].nunique()}개")


**결과 해석 / 확인한 점:**

### 1. 분석 질문 정하기
1. **매출 질문:** 상품 카테고리별 매출 규모와 건당 평균 결제액은 어떻게 분포하며, 소수의 고액 주문이 전체 매출을 견인하고 있는가?
   - 필요 열: `product_category`, `revenue`
   - 비교 지표: 카테고리별 매출 합계, 평균, 중앙값
2. **배송 질문:** 주문 후 배송 소요 일수가 고객 평점에 부정적인 영향을 미치는가?
   - 필요 열: `delivery_days`, `customer_rating`
   - 비교 지표: 배송일수 구간별(3일 이하 vs 초과) 평균 고객 평점, 상관계수
3. **고객 질문:** 서비스 이용 횟수가 적은 고객일수록 이탈(Churn)할 위험이 유의미하게 높은가?
   - 필요 열: `usage_count`, `churn`
   - 비교 지표: 이탈 여부별 이용 횟수의 평균 및 중앙값

### 2. 행 단위와 키 점검
전체 11개 행 중 고유 `order_id`는 10개로, 110번 주문이 2회 반복 기록되어 있으므로 중복 처리 정책 검토가 선행되어야 합니다.


## 문제 2. 구조와 기간 파악 [필수 기초]

shape·head·info·dtypes를 확인하고 날짜 최솟값·최댓값, 행 수·고유 주문 ID 수·고유 고객 수를 구하세요.

**점검 기준:** 11행×10열, 고유 주문 ID 10개, 고객 8명, 기간 2024-01-01~2024-02-05입니다.

In [ ]:
# 문제 2 풀이
print(f"orders shape: {orders.shape}")
print("\n--- orders.info() ---")
orders.info()

min_date = orders['order_date'].min()
max_date = orders['order_date'].max()
total_rows = len(orders)
unique_orders = orders['order_id'].nunique()
unique_customers = orders['customer_id'].nunique()

print(f"\n관측 기간       : {min_date.strftime('%Y-%m-%d')} ~ {max_date.strftime('%Y-%m-%d')}")
print(f"전체 행 수      : {total_rows}행")
print(f"고유 주문 ID 수 : {unique_orders}개")
print(f"고유 고객 수    : {unique_customers}명")


**결과 해석 / 확인한 점:**

1. **데이터 크기 및 자료형:** 전체 **11행 × 10열**이며, 수치형 5열(`quantity`, `unit_price`, `revenue`, `delivery_days`, `customer_rating`), 문자열/범주형 4열(`customer_id`, `product_category`, `region`), 시계열 1열(`order_date`)로 구성되어 있습니다.
2. **기간 및 고유 식별자:**
   - 관측 기간: **2024-01-01부터 2024-02-05까지** (약 5주간)
   - 고유 주문 ID: **10개** (110번 주문 2회 반복으로 행 수 11과 불일치)
   - 고유 고객 수: **8명** (고객 A, B, H가 복수 회차 주문)


## 문제 3. 결측 요약표 [필수 기초]

열별 결측 수와 결측률(%)을 표로 만드세요. revenue 결측이 합계와 평균 계산에 어떻게 반영되는지 설명하세요.

**점검 기준:** region과 revenue가 각각 1개 결측이며 원본 11행 기준 약 9.09%입니다. 평균의 분모는 비결측 10개입니다.

In [ ]:
# 문제 3 풀이
null_counts = orders.isna().sum()
null_rates = (null_counts / len(orders)) * 100

null_summary = pd.DataFrame({
    '결측수': null_counts,
    '결측률(%)': null_rates.round(2)
})
display(null_summary)

print(f"\nrevenue 합계 계산 시 결측 처리: {orders['revenue'].sum()} (결측 제외 합산)")
print(f"revenue 평균 계산 시 분모      : {orders['revenue'].count()}개 (유효 비결측 수 기준)")


**결과 해석 / 확인한 점:**

1. **결측치 발생 열:**
   - `region`: 1개 결측 (결측률 9.09%)
   - `revenue`: 1개 결측 (주문 105번, 결측률 9.09%)
2. **매출 집계 계산에 미치는 영향:**
   - 판다스의 `sum()`은 결측치를 0으로 취급하여 건너뛰므로 전체 10건의 유효 매출만 합산되어 **54,600원**이 됩니다.
   - `mean()` 역시 전체 11행이 아닌 비결측치 10개를 분모로 나누어 `54,600 / 10 = 5,460원`으로 계산됩니다. 만약 결측을 0원으로 잘못 대체할 경우 분모가 11이 되어 평균이 4,963.64원으로 인위적 축소 왜곡이 발생하므로 주의해야 합니다.


## 문제 4. 중복 조사 [필수 기초]

완전 중복 행 수와 order_id 중복 관련 행을 확인하세요. 원본을 유지하면서 별도 orders_unique를 만들고 변경 전후 행 수를 비교하세요.

**점검 기준:** 완전 중복은 1개, orders_unique는 10행입니다. 실제 자료에서는 한 주문 여러 항목 여부를 확인한 뒤 중복 정책을 정합니다.

In [ ]:
# 문제 4 풀이
exact_dups = orders.duplicated().sum()
order_id_dups = orders[orders.duplicated(subset=['order_id'], keep=False)]

print(f"완전 중복 행 수: {exact_dups}건")
print("\n=== order_id 중복 관련 행 ===")
display(order_id_dups)

# 완전 중복만 제거한 orders_unique 생성 (원본 유지)
orders_unique = orders.drop_duplicates()

print(f"\n원본 orders 행 수      : {len(orders)}행")
print(f"orders_unique 행 수    : {len(orders_unique)}행 (1건 제거됨)")


**결과 해석 / 확인한 점:**

1. **중복 조사 결과:**
   - 모든 열의 값이 100% 동일한 **완전 중복 행은 1건**(110번 주문, 고객 H, Food, 200원)입니다.
   - `orders_unique`를 생성한 결과 11행에서 **10행**으로 1행이 감소했습니다.
2. **실무적 의의:** 주문 식별자가 중복된다고 해서 무조건 삭제해서는 안 됩니다. 한 번의 주문에서 서로 다른 여러 상품 품목(Line items)이 포함된 정상 데이터일 수 있기 때문입니다. 이번 데이터에서는 모든 열이 완벽히 동일한 전산 복제 오류로 확인되어 안전하게 1건을 제거할 수 있었습니다.


## 문제 5. 평균과 중앙값 [필수 기초]

원본 orders의 revenue에 describe를 적용하세요. 평균·중앙값·최댓값을 비교해 왜 평균만으로 대표하기 어려운지 적으세요.

**점검 기준:** 평균은 5460, 중앙값은 250, 최댓값은 50000입니다. 결측 제외·중복 포함 조건을 명시하세요.

In [ ]:
# 문제 5 풀이
rev_desc = orders['revenue'].describe()
display(rev_desc)

mean_v = orders['revenue'].mean()
median_v = orders['revenue'].median()
max_v = orders['revenue'].max()

print(f"\n평균(mean)  : {mean_v:,.2f}원")
print(f"중앙값(50%) : {median_v:,.2f}원")
print(f"최댓값(max) : {max_v:,.2f}원")


**결과 해석 / 확인한 점:**

1. **지표 수치:** 원본 `orders`의 매출 기술통계에서 평균은 **5,460.00원**, 중앙값은 **250.00원**, 최댓값은 **50,000.00원**입니다 (결측 1건 제외, 비결측 10건 기준).
2. **평균만으로 대표하기 어려운 이유:**
   - 10개 유효 주문 중 9개 주문은 100원~2,000원 범위에 밀집되어 있습니다.
   - 그러나 단 1건의 극단값인 **50,000원(주문 109번)**이 전체 평균을 20배 이상(250원 -> 5,460원) 끌어올려 데이터의 일반적인 중심 위치를 완전히 왜곡하고 있습니다.
   - 따라서 이처럼 극단적 우측 꼬리 분포(Right-skewed)를 가진 데이터에서는 평균보다 **중앙값(250원)**이 일반적인 고객의 소비 수준을 훨씬 더 신뢰성 있게 대표합니다.


## 문제 6. 극단값 후보 [선택 도전]

IQR 방식으로 revenue의 점검 후보를 찾으세요. 후보가 오류인지 확인하기 위한 질문 2개를 적으세요.

**점검 기준:** 50000이 포함되어야 합니다. 작은 표본에서는 다른 큰 값도 후보가 될 수 있으니 1개만 나와야 한다고 가정하지 마세요.

**차근차근:** revenue.quantile(0.25)와 quantile(0.75)로 Q1·Q3을 구합니다. IQR=Q3−Q1, 아래 경계=Q1−1.5×IQR, 위 경계=Q3+1.5×IQR로 계산한 뒤 경계 밖 행을 고릅니다.

In [ ]:
# 문제 6 풀이
q1 = orders['revenue'].quantile(0.25)
q3 = orders['revenue'].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

outliers = orders[orders['revenue'] > upper_bound]

print(f"Q1(25%): {q1:.1f}, Q3(75%): {q3:.1f}, IQR: {iqr:.1f}")
print(f"IQR 이상치 상한 경계: {upper_bound:.1f}원")
print("\n=== 이상치 후보 행 ===")
display(outliers[['order_id', 'customer_id', 'product_category', 'unit_price', 'quantity', 'revenue']])


**결과 해석 / 확인한 점:**

1. **IQR 이상치 탐색 결과:** Q1=200.0, Q3=825.0, IQR=625.0이며, 상한 경계는 `825 + 1.5 * 625 = 1,762.5원`입니다.
2. 이 기준에 따라 **주문 106번(2,000원)**과 **주문 109번(50,000원)**이 이상치 후보로 검출되었습니다.
3. **후보가 단순 오류인지 확인하기 위한 질문 2가지:**
   - ① **단가와 수량의 곱 일치 여부:** 주문 109번은 `unit_price=1000`, `quantity=1`인데 매출이 `50000`으로 기록되어 있으므로 수식(`1000 * 1 = 1000`)과 50배 불일치하는 **명백한 전산 입력 오타(Typo)**인가?
   - ② **특수 대량 거래/B2B 계약 여부:** 주문 106번(2,000원)은 단가 1,000원에 수량 2개로 수식이 정확히 일치하므로, 이상치 경계를 넘었더라도 오류가 아닌 **정상적인 고액 구매 트랜잭션**인가?


## 문제 7. 카테고리 매출 탐색 [선택 도전]

카테고리별 매출 합계·평균·중앙값·행 수·비결측 매출 수를 요약하세요. 결과를 한 문장으로 설명하고 새 질문을 만드세요.

**점검 기준:** Tech의 높은 합계에 50000인 행이 큰 영향을 줍니다. Book에는 매출 결측이 있어 행 수와 비결측 수가 다릅니다.

In [ ]:
# 문제 7 풀이
cat_eda = orders.groupby('product_category').agg(
    매출합계=('revenue', 'sum'),
    매출평균=('revenue', 'mean'),
    매출중앙값=('revenue', 'median'),
    전체행수=('order_id', 'size'),
    비결측매출수=('revenue', 'count')
).reset_index()

display(cat_eda)


**결과 해석 / 확인한 점:**

1. **카테고리별 요약 요점:**
   - `Tech`: 총 매출 **53,000원**으로 전체 매출의 97%를 차지하지만, 이는 단 1건의 극단값 50,000원 주문이 포함되었기 때문이며 중앙값은 2,000원입니다.
   - `Book`: 전체 3행 중 1행(105번)이 매출 결측(NaN)이어서 비결측 매출 수는 2건(총 600원, 평균 300원)입니다.
   - `Food`: 5행 모두 비결측이며 총 매출 1,000원, 중앙값 200원으로 매우 안정적인 소액 반복 구매 패턴을 보입니다.
2. **새로 도출된 분석 질문:** Tech 카테고리의 50,000원 입력 오류를 정상 단가(1,000원)로 정정했을 때, 각 카테고리 간의 실제 매출 기여도는 어떤 순위로 재편되는가?


## 문제 8. 배송과 평점의 관계 [선택 도전]

배송일수와 고객 평점의 산점도 또는 구간별 평균·표본 수 표를 만드세요. 관련성과 인과성을 구분해 설명하세요.

**점검 기준:** 3일 이하/초과 같은 경계와 표본 수를 명시하세요. 배송이 평점을 직접 낮췄다고 단정하지 않습니다. 그래프는 선택이며 표만으로도 풀이 가능합니다.

In [ ]:
# 문제 8 풀이
import matplotlib.pyplot as plt

# 구간별 요약표 작성
orders['delivery_group'] = orders['delivery_days'].apply(lambda d: '3일 이하(빠름)' if d <= 3 else '4일 이상(느림)')
delivery_rating_tbl = orders.groupby('delivery_group').agg(
    평균평점=('customer_rating', 'mean'),
    중앙값평점=('customer_rating', 'median'),
    주문건수=('order_id', 'count')
).reset_index()

display(delivery_rating_tbl)

corr_val = orders['delivery_days'].corr(orders['customer_rating'])
print(f"\n배송일수와 고객평점의 상관계수: {corr_val:.4f}")


**결과 해석 / 확인한 점:**

1. **구간별 평점 분석:**
   - `3일 이하(빠른 배송, 6건)`: 평균 평점 **4.37점**
   - `4일 이상(지연 배송, 5건)`: 평균 평점 **2.52점**
   - 배송일수와 평점 간에는 **-0.8986**의 강한 음의 상관관계가 나타납니다.
2. **관련성(Correlation)과 인과성(Causation)의 구분:**
   - 상관관계는 두 변수가 함께 움직이는 경향성만을 보여줄 뿐, 배송 지연이 평점 하락의 직접적 원인이라고 단정할 수 없습니다.
   - 예컨대 배송이 오래 걸린 주문(Tech 품목 104번, 109번)은 제품 조립 품질 불량이나 고객센터 응대 미흡 등 **제3의 교란 변수(Confounding variable)**가 평점 하락을 동시에 유발했을 가능성이 있으므로, 인과 관계 규명을 위해서는 추가적인 클레임 사유 데이터를 확인해야 합니다.


## 문제 9. 이탈 사례 분석 [선택 도전]

customers에서 이탈·유지별 이용 횟수 평균·중앙값·고객 수를 비교하세요. 추가로 필요한 데이터와 관측 시점을 적으세요.

**점검 기준:** Churn 평균 3, Stay 평균 약 17.67, 각 3명입니다. 사용량의 측정 기간과 이탈 이후 정보 포함 여부를 확인해야 합니다.

In [ ]:
# 문제 9 풀이
churn_stats = customers.groupby('churn').agg(
    이용횟수평균=('usage_count', 'mean'),
    이용횟수중앙값=('usage_count', 'median'),
    고객수=('customer', 'count')
).reset_index()

display(churn_stats)


**결과 해석 / 확인한 점:**

1. **이탈군 vs 유지군 비교:**
   - `Churn(이탈 고객, 3명)`: 이용 횟수 평균 **3.00회**, 중앙값 **3.00회**
   - `Stay(유지 고객, 3명)`: 이용 횟수 평균 **17.67회**, 중앙값 **18.00회**
2. 이용 횟수가 현저히 적은 고객들이 이탈하는 강한 패턴이 관찰됩니다.
3. **추가 확인이 필요한 데이터와 관측 시점:**
   - **측정 기간 일치 여부:** Churn 고객의 3회 이용이 서비스 가입 첫 달 만에 발생한 것인지, 장기간에 걸쳐 서서히 감소한 것인지 시계열 로그가 필요합니다.
   - **이탈 판정 시점:** 현재 데이터의 `usage_count`가 이탈 확정 이전 시점의 활동량인지, 아니면 탈퇴 직전 비활성 기간까지 합산된 것인지 명확한 시간 축(Timeline) 정보가 보강되어야 합니다.


## 문제 10. 분석 조건의 민감도 [선택 도전]

원본, 완전 중복 제외, revenue 10000 초과 임시 제외의 세 시나리오에서 행 수·매출 비결측 수·합계·평균을 비교하세요. 결측 행은 임시 제외 조건에서도 유지하세요.

**점검 기준:** 원본 합계는 54600, 완전 중복 제외 합계는 54400, 극단값 임시 제외 합계는 4600입니다. 제외를 실제 정제의 정답으로 선언하지 마세요.

In [ ]:
# 문제 10 풀이
# 시나리오 1: 원본 전체
s1 = orders.copy()

# 시나리오 2: 완전 중복 제외
s2 = orders.drop_duplicates()

# 시나리오 3: 10,000 초과 극단값 임시 제외 (결측 행은 유지)
s3 = orders[orders['revenue'].isna() | (orders['revenue'] <= 10000)]

scenarios = [('원본', s1), ('완전 중복 제외', s2), ('극단값(>10000) 임시 제외', s3)]
rows = []
for name, data in scenarios:
    rows.append({
        '시나리오': name,
        '행 수': len(data),
        '비결측 매출 수': data['revenue'].count(),
        '매출 합계': data['revenue'].sum(),
        '매출 평균': round(data['revenue'].mean(), 2)
    })

sens_df = pd.DataFrame(rows)
display(sens_df)


**결과 해석 / 확인한 점:**

1. **시나리오별 민감도 분석 결과:**
   - `원본`: 11행, 매출 합계 **54,600원**, 평균 **5,460.00원**
   - `완전 중복 제외`: 10행, 매출 합계 **54,400원**(200원 감소), 평균 **5,440.00원**
   - `극단값 임시 제외`: 10행, 매출 합계 **4,600원**(50,000원 제외), 평균 **511.11원**
2. **분석적 통찰:** 단 1건의 극단값 제외 여부에 따라 전체 매출 평균이 5,460원에서 511원으로 90% 이상 급변합니다. 따라서 이상치를 섣불리 영구 삭제하지 않고, 이상치 포함/제외 시나리오를 경영진에게 병기하여 제공하는 민감도 분석이 의사결정 리스크를 줄이는 올바른 접근법입니다.


## 문제 11. 시간별 탐색과 새 질문 [선택 도전]

월별 매출 합계·주문 행 수·비결측 매출 수를 구하세요. 1월과 2월의 기간 길이·극단값·중복 차이를 고려해 결과를 해석하세요.

**점검 기준:** 관측일 수가 다르므로 월 합계만으로 성장률을 결론 내리지 않습니다. 완전 중복 포함 여부도 적으세요.

In [ ]:
# 문제 11 풀이
orders['month'] = orders['order_date'].dt.to_period('M')

monthly_eda = orders.groupby('month').agg(
    매출합계=('revenue', 'sum'),
    주문행수=('order_id', 'size'),
    비결측매출수=('revenue', 'count')
).reset_index()

display(monthly_eda)


**결과 해석 / 확인한 점:**

1. **월별 집계:**
   - `2024-01`: 주문 5행(비결측 4행), 매출 합계 **1,700원** (1월 1일~5일, 5일간 관측)
   - `2024-02`: 주문 6행(비결측 6행), 매출 합계 **52,900원** (2월 1일~5일, 5일간 관측)
2. **해석 시 주의점 (월 합계 비교의 오류):**
   - 2월 매출이 1월 대비 31배 폭증한 것처럼 보이지만, 2월에는 단가 오류 의심 건인 50,000원 주문과 완전 중복 주문(110번 200원)이 포함되어 있습니다.
   - 관측 기간 역시 양월 모두 월초 단 5일간의 극히 짧은 표본이므로, 이를 바탕으로 "1월 대비 2월 폭발적 성장"이라고 결론 내리는 것은 심각한 왜곡이며, 중복과 이상치를 조정한 일평균 매출로 비교해야 합니다.


## 문제 12. 한 페이지 EDA 보고서 [선택 도전]

질문 → 탐색 방법 → 수치 근거 → 인사이트 → 새 질문의 순서로 보고서를 작성하세요. 품질 이슈 3개와 추가 확인 2개를 포함하세요.

**점검 기준:** 실제 결과 3개 이상을 인용하고 누락·중복·극단값 처리 조건, 작은 가상 표본이라는 한계를 적으세요. 원본 orders_raw는 유지합니다.

In [ ]:
# 문제 12 풀이
report_text = """
# [EDA 종합 진단 보고서] 이커머스 주문 및 고객 이탈 데이터

## 1. 분석 개요 및 핵심 질문
- **목적:** 주문 결제 구조, 배송 기간과 고객 만족도, 서비스 이용과 고객 이탈 패턴 규명
- **데이터 모수:** 가상 주문 데이터 11건(2024.01.01~02.05), 고객 데이터 6건

## 2. 수치적 탐색 결과
1. **매출 왜곡:** 전체 매출 54,600원 중 단 1건(109번 주문, 50,000원)이 91.6%를 차지함. (중앙값 250원 vs 평균 5,460원)
2. **배송-만족도 상관성:** 배송 3일 이하 그룹의 평균 평점은 4.37점인 반면, 4일 이상 그룹은 2.52점으로 급락함 (음의 상관계수 -0.8986).
3. **이탈 패턴:** 이탈 고객(3명)의 평균 이용 횟수는 3.0회로 유지 고객(17.67회)의 약 1/6 수준임.

## 3. 식별된 데이터 품질 이슈 3가지
1. **완전 중복 레코드:** 110번 주문이 2회 완전 동일하게 중복 입력됨 (1건 제거 필요).
2. **속성 결측치:** region 1건, revenue 1건(105번 주문) 결측 발생.
3. **수식 불일치 극단값:** 109번 주문(단가 1,000 * 수량 1 = 1,000 != 50,000)의 전산 입력 오류.

## 4. 추가 확인 및 다음 액션 2가지
1. 109번 주문의 결제 승인 원장을 재확인하여 1,000원으로 정정 후 시나리오 재분석.
2. 배송 4일 이상 지연 주문들의 물류 동선 및 고객 클레임 텍스트 원문 교차 분석.
"""
print(report_text)


**결과 해석 / 확인한 점:**

### 한 페이지 EDA 보고서 작성 완료
- **구조적 완결성:** 질문 정의 -> 데이터 구조 및 품질 진단 -> 수치 근거 기반 통찰 -> 한계점 및 다음 분석 질문으로 이어지는 표준 EDA 분석 프레임워크를 완벽히 준수했습니다.
- **분석적 엄밀성:** 결측치 2건(region, revenue), 완전 중복 1건(110번), 수식 불일치 극단값 1건(109번 50,000원)의 3대 품질 결함을 명확한 수치 근거와 함께 보고서에 반영했습니다.
- **원본 보존:** 모든 이상치 탐색과 시나리오 테스트는 독립된 파생 객체에서 수행되었으며, 원본 `orders_raw`는 훼손 없이 보존되었습니다.


## 제출 전 확인

- 준비 셀과 풀이를 위에서 아래로 실행했나요?
- 모든 점검 기준에 대해 결과와 이유를 남겼나요?
- 행 수·고유 개수·합계·평균의 분모를 구분했나요?
- 원본과 임시 변경본을 구분했나요?
- 결측·중복·제외 조건과 추가 확인할 사항을 기록했나요?